# QASM ↔ Cirq round-trip smoke tests

Exercises `src/cirq_qasm.py`:

- `qasm_to_cirq` / `qasm_str_to_cirq` — load OpenQASM 2.0 → `cirq.Circuit`
- `cirq_to_qasm` / `cirq_to_qasm_str` — serialize `cirq.Circuit` → OpenQASM 2.0

We verify **functional** equivalence (unitary / final-state equality) rather than
textual equivalence, because Cirq may decompose gates during the round-trip.

In [1]:
import sys, os
sys.path.append(os.path.abspath('src'))

import numpy as np
import cirq
from qiskit import QuantumCircuit, qasm2
from qiskit.quantum_info import Operator

from cirq_qasm import (
    qasm_to_cirq, qasm_str_to_cirq,
    cirq_to_qasm, cirq_to_qasm_str,
    QasmVersionError,
)

## Test 1 — Qiskit → QASM2 file → Cirq

Build a small circuit in Qiskit, dump to `test_circuit.qasm`, load it back with
`qasm_to_cirq`. Verify the Cirq circuit has the same number of qubits and at
least the expected operations.

In [2]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.x(1)
qc.z(0)
# Note: no measure_all() — measurements complicate unitary comparison below.

print('Starting Qiskit circuit:')
print(qc.draw(output='text'))

qasm_path = 'test_circuit.qasm'
with open(qasm_path, 'w') as fd:
    qasm2.dump(qc, fd)

cirq_circ = qasm_to_cirq(qasm_path)

assert len(cirq_circ.all_qubits()) == 2, 'qubit count mismatch'
print('\nEnding Cirq circuit (after QASM round-trip):')
print(cirq_circ)

Starting Qiskit circuit:
     ┌───┐     ┌───┐
q_0: ┤ H ├──■──┤ Z ├
     └───┘┌─┴─┐├───┤
q_1: ─────┤ X ├┤ X ├
          └───┘└───┘

Ending Cirq circuit (after QASM round-trip):
q_0: ───H───@───Z───
            │
q_1: ───────X───X───


## Test 2 — Functional equivalence: Qiskit unitary vs Cirq unitary

The strongest round-trip check. If Cirq decomposed some gates, the unitaries
should still match (up to numerical noise). `np.allclose` with atol=1e-10 is
the bar.

In [3]:
u_qiskit = Operator(qc).data              # Qiskit's 4x4 unitary
u_cirq   = cirq.unitary(cirq_circ)         # Cirq's 4x4 unitary

# Qiskit and Cirq use opposite qubit-ordering conventions — Qiskit is little-endian,
# Cirq is big-endian. For a 2-qubit comparison we reorder one of them.
def swap_endian_2q(u):
    perm = [0, 2, 1, 3]  # swap qubit order for 2 qubits
    return u[np.ix_(perm, perm)]

match_direct    = np.allclose(u_qiskit, u_cirq, atol=1e-10)
match_swapped   = np.allclose(u_qiskit, swap_endian_2q(u_cirq), atol=1e-10)

print(f'unitary match (direct):     {match_direct}')
print(f'unitary match (swap-order): {match_swapped}')
assert match_direct or match_swapped, 'round-trip changed the unitary'

unitary match (direct):     False
unitary match (swap-order): True


## Test 3 — Cirq → QASM string → Cirq (pure round-trip)

Build a circuit directly in Cirq, serialize, re-load, compare unitaries.
No Qiskit involved — isolates whether our wrappers preserve semantics.

In [4]:
q0, q1 = cirq.LineQubit.range(2)
original = cirq.Circuit(
    cirq.H(q0),
    cirq.CX(q0, q1),
    cirq.rx(0.37)(q1),
    cirq.rz(1.21)(q0),
)

print('Starting Cirq circuit:')
print(original)

qasm_text = cirq_to_qasm_str(original)
round_trip = qasm_str_to_cirq(qasm_text)

print('\nEnding Cirq circuit (after QASM round-trip):')
print(round_trip)

u_original   = cirq.unitary(original)
u_round_trip = cirq.unitary(round_trip)

print('\nQASM output (first 12 lines):')
print('\n'.join(qasm_text.splitlines()[:12]))
print()
print(f'unitary match: {np.allclose(u_original, u_round_trip, atol=1e-10)}')
assert np.allclose(u_original, u_round_trip, atol=1e-10), 'round-trip changed the unitary'

Starting Cirq circuit:
0: ───H───@───Rz(0.385π)───
          │
1: ───────X───Rx(0.118π)───

Ending Cirq circuit (after QASM round-trip):
q_0: ───H───@───Rz(0.385π)───
            │
q_1: ───────X───Rx(0.118π)───

QASM output (first 12 lines):
// Generated from Cirq v1.6.1

OPENQASM 2.0;
include "qelib1.inc";


// Qubits: [q(0), q(1)]
qreg q[2];


h q[0];
cx q[0],q[1];

unitary match: True


## Test 4 — Write Cirq → QASM file and reload

Covers `cirq_to_qasm(circuit, out_path=...)` writing to disk.

In [5]:
out_path = 'cirq_exported.qasm'

print('Starting Cirq circuit:')
print(original)

written = cirq_to_qasm(original, out_path=out_path)

assert os.path.isfile(out_path), 'file was not written'
reloaded = qasm_to_cirq(out_path)

print('\nEnding Cirq circuit (reloaded from disk):')
print(reloaded)

assert np.allclose(cirq.unitary(original), cirq.unitary(reloaded), atol=1e-10), \
    'disk round-trip changed the unitary'
print(f'\nwrote {out_path} ({len(written)} chars), reload verified.')

Starting Cirq circuit:
0: ───H───@───Rz(0.385π)───
          │
1: ───────X───Rx(0.118π)───

Ending Cirq circuit (reloaded from disk):
q_0: ───H───@───Rz(0.385π)───
            │
q_1: ───────X───Rx(0.118π)───

wrote cirq_exported.qasm (180 chars), reload verified.


## Test 5 — Error paths

Unsupported QASM 3 input and a missing file both raise clearly.

In [6]:
qasm3_sample = 'OPENQASM 3.0;\ninclude "stdgates.inc";\nqubit[1] q;\nh q[0];\n'

try:
    qasm_str_to_cirq(qasm3_sample)
except QasmVersionError as e:
    print(f'PASS (QASM3 rejected): {e}')
else:
    raise AssertionError('QASM3 should have been rejected')

try:
    qasm_to_cirq('nonexistent_file.qasm')
except FileNotFoundError as e:
    print(f'PASS (missing file rejected): {e}')
else:
    raise AssertionError('missing file should have raised')

PASS (QASM3 rejected): Cirq's QASM importer supports OpenQASM 2.0 only; got OpenQASM 3. Re-export your source with qasm2.dump / qasm2.dumps instead of qasm3.
PASS (missing file rejected): QASM file not found: nonexistent_file.qasm


In [7]:
# Optional: remove test artifacts
for f in ('test_circuit.qasm', 'cirq_exported.qasm'):
    if os.path.exists(f):
        os.remove(f)
print('All tests passed.')

All tests passed.
